# Adult Census Income: feature engineering

В этом notebook рассматриваем согласованные engineered features.

Добавляем:

- `is_married`;
- `native_country_group`;
- `workclass_group`;
- `hours_per_week_bin`;
- `capital_net`;
- `has_capital_gain`;
- `has_capital_loss`.

Исходные признаки не удаляем. Новые признаки добавляются рядом с исходными, чтобы модель могла использовать и детальную категорию, и более обобщённую информацию.

## Загрузка очищенных данных и добавление признаков

Загрузка, базовая очистка и функция `add_features` вынесены в `adult_income_utils.py`, чтобы не дублировать код между notebook.

In [ ]:
import numpy as np
import pandas as pd

import sys
from pathlib import Path

sys.path.append(str(Path.cwd()))
sys.path.append(str(Path.cwd() / "src"))

from adult_income_utils import add_features, load_clean_adult_data

pd.set_option("display.max_columns", 100)

In [ ]:
df_clean = load_clean_adult_data()
df_features = add_features(df_clean)

print("Rows after cleaning:", len(df_clean))
print("Shape after feature engineering:", df_features.shape)

df_features.head()

## 1. Признак `is_married`

Исходный `marital_status` содержит несколько категорий. Для модели может быть полезен более общий бинарный признак: состоит ли человек в браке сейчас.

К группе `1` относим:

- `Married-civ-spouse`;
- `Married-AF-spouse`.

Все остальные статусы относим к `0`.

In [ ]:
df_features[["marital_status", "is_married"]].drop_duplicates().sort_values("marital_status")

## 2. Признак `native_country_group`

`native_country` содержит много стран, при этом основная категория — `United-States`. Редкие страны объединяем в группу `Other`, а пропуски сохраняем как `Unknown`.

Такой признак не заменяет исходную страну, а добавляет более устойчивую обобщённую информацию.

In [ ]:
df_features["native_country_group"].value_counts()

## 3. Признак `workclass_group`

`workclass` группируем по типу занятости:

- `Private`;
- `Government`;
- `Self-employed`;
- `Other`;
- `Unknown`.

Это снижает детализацию, но делает признак более компактным и интерпретируемым.

In [ ]:
pd.crosstab(df_features["workclass"].fillna("Unknown"), df_features["workclass_group"])

## 4. Признак `hours_per_week_bin`

`hours_per_week` оставляем как числовой признак, но дополнительно добавляем категориальную группировку по рабочим часам:

- `part_time` — до 34 часов в неделю;
- `full_time_40` — от 35 до 40 часов;
- `over_40` — от 41 до 50 часов;
- `extreme` — больше 50 часов.

Такой признак помогает явно выделить стандартную, неполную и повышенную занятость.

In [ ]:
pd.crosstab(df_features["hours_per_week_bin"], df_features["income"], normalize="index").round(3)

## 5. Capital-признаки

`capital_gain` и `capital_loss` у большинства объектов равны нулю. Поэтому полезно добавить признаки, которые показывают не только размер capital gain/loss, но и сам факт их наличия.

Добавляем:

- `capital_net = capital_gain - capital_loss`;
- `has_capital_gain`;
- `has_capital_loss`.

Исходные признаки `capital_gain` и `capital_loss` сохраняем.

In [ ]:
df_features[["capital_gain", "capital_loss", "capital_net", "has_capital_gain", "has_capital_loss"]].head()

## Проверка результата

In [ ]:
new_features = [
    "is_married",
    "native_country_group",
    "workclass_group",
    "hours_per_week_bin",
    "capital_net",
    "has_capital_gain",
    "has_capital_loss",
]

df_features[new_features].head()

In [ ]:
print("Original shape:", df_clean.shape)
print("Shape after feature engineering:", df_features.shape)

df_features[new_features].isna().sum()

## Решение

Пока оставляем эти engineered features:

- `is_married` — простой бинарный признак семейного статуса;
- `native_country_group` — обобщение страны происхождения до `United-States`, `Other`, `Unknown`;
- `workclass_group` — обобщение типа занятости;
- `hours_per_week_bin` — группировка по рабочим часам в неделю;
- `capital_net` — разница между `capital_gain` и `capital_loss`;
- `has_capital_gain` — факт наличия capital gain;
- `has_capital_loss` — факт наличия capital loss.

Исходные признаки не удаляем. Позже можно сравнить качество моделей с engineered features и без них.